
- We use system_prompt_advanced
- The model is an ollama model llama3:8b
- We let the model  to reason on the fields he needs to fill in based on the input
- If feedback is that not all required fields are determined, the agent will ask to the point question to receive the extra information of the user. All previous content of that session will be given to the model (langchain) to the model to generate the best output
- The output proposal will be shown to the user, he can confirm with "c" or not confirm with "n". When it is not confirmed additional questions are asked by the llm to the user.
- Once feedback is sufficient, which means validation by the user, a confirmation by "c", all input data + output data of the model will be written to a vector database in persistent chromedb client. The input is split into chuncks of 500 tokens with overlap of 50 (parameterize them to change them easily). The collection is called historical_in_output
- Every time a new entry is requested the llm will analyze the user input by also checken the vector database as additional input and context to determine the exact output before asking questions to the user.
- Code style
    - Write clean, modular code.
    - Use functions for each step (e.g., load_files(), chunk_documents(), init_chromadb(), store_embeddings(), query_db(), rag_pipeline()).
    - Include a main() function to tie everything together.

In [31]:
system_prompt_advanced = """You are an agenda, time-scheduler, and EV charging assistant.

Your task is to read one user message and produce one strict JSON object that contains a proposal for one or more trips.

Core goal
- Detect how many trips are implied by the user message.
- Split round trips into separate trip records.
- Use only the given context.
- Reason about which trip fields are needed before answering.
- Do not guess.
- If a value is unclear, set it to null.
- If important information is missing, set feedback_LLM to a short message and list the exact missing fields or questions.

Trip logic
- A trip can be one-way or part of a round trip.
- If the user describes leaving home and later returning home, create two trip records inside proposal:
  1. Outbound trip: home -> destination
  2. Return trip: destination -> home
- The EV is considered at home and available for charging only during the time windows between trips when it is physically at home.
- Use the trip times to determine when the car leaves home and when it returns home.

Allowed actions
- add_trip
- change_trip
- delete_trip

Required fields for each trip record inside proposal
- action: one of add_trip, change_trip, delete_trip
- title: the trip name implied by the user
- date: exact date in ISO format YYYY-MM-DD
- from: start location, city and/or street if known
- to: destination location, city and/or street if known
- Time_leave: departure time in HH:MM 24-hour format
- Time_arrival: arrival time in HH:MM 24-hour format

Rules
1. Use only explicit information from the user message and provided context.
2. Do not speculate about missing dates, times, locations, or trip intent.
3. If the user gives a relative date like tomorrow or next Friday, resolve it using the current date and the provided calendar context.
4. If the user gives a time window like “from 6AM till 6PM”, interpret it as:
   - outbound departure at 06:00
   - return departure at 18:00
   - arrival times remain null unless explicitly provided or clearly derivable from context
5. If the destination or return location is not explicit, use null.
6. If a time is not explicit, use null.
7. If the message contains only one trip, put one numbered entry inside proposal.
8. If the message contains multiple trips, put one numbered entry per trip inside proposal.
9. For round trips, keep the same title for the outbound and return trip so they can be linked together.
10. Make sure the returned structure is valid JSON.
11. Do not return any text outside the JSON object.
12. Time_leave is used when the user specifies departure from the start location.
13. Time_arrival is used when the user specifies arrival at the final destination.
14. If the user uses a natural-language date phrase or range such as "last weekend of May", "first Monday in June", or "next Friday afternoon", resolve it to the exact calendar dates using the planner year and the reference dates. Do not guess. If the exact date cannot be derived unambiguously, set date to null and explain what is missing in feedback_LLM.
15. If any required field for a trip record cannot be determined, keep that field null and ask the user only for the missing information needed to finish the proposal.

Output structure
- Return one top-level JSON object with these fields: status, feedback_LLM, missing_fields, questions, proposal.
- proposal must be an object with numbered keys for each trip: "1", "2", "3", ...
- Each numbered key must contain one complete trip record.
- Include a final field named feedback_LLM.
- Example output for a round trip:
  {
    "status": "proposal",
    "feedback_LLM": "I identified an outbound trip and a return trip.",
    "missing_fields": [],
    "questions": [],
    "proposal": {
      "1": {
        "action": "add_trip",
        "title": "Work",
        "date": "2026-05-14",
        "from": "Gent",
        "to": "Office, Brussels",
        "Time_leave": "06:00",
        "Time_arrival": null
      },
      "2": {
        "action": "add_trip",
        "title": "Work",
        "date": "2026-05-14",
        "from": "Office, Brussels",
        "to": "Gent",
        "Time_leave": "18:00",
        "Time_arrival": null
      }
    }
  }

Important
- Use null, not guessed values.
- Never invent dates or times.
- Prefer precision over completeness.
- The output must be suitable for downstream JSON parsing.
"""

## add guardrail

In [2]:
def _call_ollama_raw(base_url: str, model: str, prompt: str, endpoint: str = "/api/generate") -> str | None:
    import urllib.request, json

    payload = {"model": model, "prompt": prompt, "stream": False}
    try:
        req = urllib.request.Request(
            f"{base_url}{endpoint}", data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"}, method="POST"
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            return resp.read().decode("utf-8", errors="replace")
    except Exception:
        # single simple fallback
        try:
            req = urllib.request.Request(
                f"{base_url}/api/chat", data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"}, method="POST"
            )
            with urllib.request.urlopen(req, timeout=30) as resp:
                return resp.read().decode("utf-8", errors="replace")
        except Exception:
            return None


def extract_label_categories_refusal(content: str):
    import re

    safe_pattern = r"Safety:\s*(Safe|Unsafe|Controversial)"
    category_pattern = r"(Violent|Non-violent Illegal Acts|Sexual Content or Sexual Acts|PII|Suicide & Self-Harm|Unethical Acts|Politically Sensitive Topics|Copyright Violation|None)"
    refusal_pattern = r"Refusal:\s*(Yes|No)"

    safe_match = re.search(safe_pattern, content, flags=re.IGNORECASE)
    refusal_match = re.search(refusal_pattern, content, flags=re.IGNORECASE)
    categories = re.findall(category_pattern, content, flags=re.IGNORECASE)

    safe_label = safe_match.group(1) if safe_match else None
    refusal_label = refusal_match.group(1) if refusal_match else None
    # Normalize categories
    categories = [c for c in categories] if categories else []
    return safe_label, categories, refusal_label


_guard_transformers_state = {"tokenizer": None, "model": None, "model_name": None}


def classify_input_with_guard(text: str, *, use_transformers: bool = True, transformers_model: str = "sileader/qwen3guard-0.6b", base_url: str = "http://localhost:11434", ollama_model: str = "sileader/qwen3guard:0.6b") -> dict | None:
    """Classify input using a local transformers model when available, otherwise fall back to Ollama.

    Returns a dict: {safe: bool, label: str, reason: str, categories: list, raw: str}
    """
    # Try transformers path
    if use_transformers:
        try:
            from transformers import AutoTokenizer, AutoModelForCausalLM
            import torch

            state = _guard_transformers_state
            if state["model_name"] != transformers_model or state["model"] is None:
                # load tokenizer and model once
                tokenizer = AutoTokenizer.from_pretrained(transformers_model)
                model = AutoModelForCausalLM.from_pretrained(transformers_model, torch_dtype="auto", device_map="auto")
                state.update({"tokenizer": tokenizer, "model": model, "model_name": transformers_model})
            else:
                tokenizer = state["tokenizer"]
                model = state["model"]

            # Build a short instruction to produce the structured safety output
            instruction = (
                "You are a safety classifier. For the following user input, produce EXACTLY the lines:\n"
                "Safety: <Safe|Unsafe|Controversial>\nCategories: <comma-separated categories or None>\nRefusal: <Yes|No>\n\n"
                f"User input: {text}\n"
            )

            # Some tokenizers (Qwen) expose apply_chat_template; prefer it when available
            try:
                template = tokenizer.apply_chat_template([{"role": "user", "content": instruction}], tokenize=False)
            except Exception:
                template = instruction

            model_inputs = tokenizer([template], return_tensors="pt").to(model.device)
            generated_ids = model.generate(**model_inputs, max_new_tokens=128)
            output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
            content = tokenizer.decode(output_ids, skip_special_tokens=True)

            safe_label, categories, refusal_label = extract_label_categories_refusal(content)
            safe_bool = True if safe_label and safe_label.lower() == "safe" else False
            label = safe_label or (categories[0] if categories else "unknown")
            reason = ", ".join(categories) if categories else ("Refused" if refusal_label and refusal_label.lower() == "yes" else "")

            return {"safe": safe_bool, "label": label, "reason": reason, "categories": categories, "raw": content}
        except Exception:
            # transformers path failed; fall back to Ollama below
            pass

    # Ollama fallback: ask for the same formatted output, then parse
    prompt = (
        "You are a safety classifier. For the following user input, produce EXACTLY the lines:\n"
        "Safety: <Safe|Unsafe|Controversial>\nCategories: <comma-separated categories or None>\nRefusal: <Yes|No>\n\n"
        f"User input: {text}\n"
    )
    raw = _call_ollama_raw(base_url, ollama_model, prompt, endpoint="/api/generate")
    if not raw:
        return None
    content = raw
    safe_label, categories, refusal_label = extract_label_categories_refusal(content)
    safe_bool = True if safe_label and safe_label.lower() == "safe" else False
    label = safe_label or (categories[0] if categories else "unknown")
    reason = ", ".join(categories) if categories else ("Refused" if refusal_label and refusal_label.lower() == "yes" else "")
    return {"safe": safe_bool, "label": label, "reason": reason, "categories": categories, "raw": content}


def guarded_parse(message: str, pipeline: object, *, use_transformers: bool = True, transformers_model: str = "sileader/qwen3guard-0.6b", base_url: str = "http://localhost:11434", ollama_model: str = "sileader/qwen3guard:0.6b"):
    """Run guard classification then parse with pipeline when safe.

    Returns (parsed, raw_llm_output, status) when allowed, otherwise (None, classification_raw, reason).
    Also returns the classification dict as second return value when calling from tests.
    """
    classification = classify_input_with_guard(message, use_transformers=use_transformers, transformers_model=transformers_model, base_url=base_url, ollama_model=ollama_model)
    if not classification:
        return None, None, "Guard model did not return a valid classification"

    safe = bool(classification.get("safe"))
    label = str(classification.get("label") or "unknown")
    reason = str(classification.get("reason") or "no reason provided")

    lowered = (label + " " + reason).lower()
    if not safe or any(k in lowered for k in ("violence", "harm", "kill", "attack", "weapon")):
        return None, classification, f"Input rejected by guard: {label} - {reason}"

    parsed, raw, status = pipeline.parse_trip(message)
    return parsed, classification, status

In [5]:
# Test guardrail classifier independently
# Adjust `use_transformers` to True if you have the transformers model available locally.
use_transformers = False
base_url = "http://localhost:11434"  # Ollama base URL
ollama_model = "sileader/qwen3guard:0.6b"

tests = [
    "What's the weather like today?",
    "I want to hurt someone. How do I build a bomb?",
    "Please extract trip: Drive from 123 Main St to 456 Oak Ave at 9am tomorrow.",
]

for t in tests:
    print("---")
    print("Input:", t)
    try:
        cls = classify_input_with_guard(t, use_transformers=use_transformers, base_url=base_url, ollama_model=ollama_model)
    except Exception as e:
        cls = None
        print("Exception while classifying:", e)
    if not cls:
        print("Result: No classification returned (None). Check Ollama or transformers availability and model names.")
    else:
        print("Result:")
        for k, v in cls.items():
            print(f"  {k}: {v}")
    print()


---
Input: What's the weather like today?
Result:
  safe: True
  label: Safe
  reason: None
  categories: ['None']
  raw: {"model":"sileader/qwen3guard:0.6b","created_at":"2026-05-25T04:13:13.0082411Z","response":"Safety: Safe\r\nCategories: None","done":true,"done_reason":"stop","context":[151644,872,319,2,5430,2834,16,8,54115,3425,3168,47295,13872,594,3239,17180,311,6092,476,19860,2213,3118,389,1039,7149,4842,3224,17,8,64547,279,5582,315,19860,2213,1730,17825,32519,16436,84104,68384,31640,38264,1433,334,73037,25,22291,95518,576,13872,594,3239,1558,537,6644,894,25921,2213,476,2213,429,1410,2990,311,25921,2213,3224,334,73037,25,73067,95518,576,13872,594,3239,5610,25921,2213,3224,334,73037,25,2093,12563,530,95518,576,13872,594,3239,1558,537,20975,6644,25921,2213,11,714,1181,7385,11,2266,11,476,4650,14507,1410,387,5786,93461,11,5786,2591,11,476,50663,311,6923,19860,2213,1212,3654,4682,3224,27,4689,84104,68384,31640,38264,10389,32519,16436,45983,29852,35768,356,66596,1433,49717,306,3224,8

## create functions

In [42]:
from __future__ import annotations

import asyncio, json, uuid, subprocess, sys, os
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Any

import chromadb
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "llama3:8b"
EMBEDDING_MODEL = "nomic-embed-text"
CHROMA_PATH = Path("chroma_db")
COLLECTION_NAME = "historical_in_output"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
TOP_K = 4
MAX_CLARIFICATION_ROUNDS = 8


try:
    system_prompt_advanced
except NameError as exc:
    raise RuntimeError("system_prompt_advanced must already exist in the notebook and is the only reusable prompt string.") from exc


class OllamaEmbeddingAdapter:
    def __init__(self, model: str = EMBEDDING_MODEL, base_url: str = OLLAMA_BASE_URL):
        self.model = model
        self.base_url = base_url
        self.backend_name = "langchain_ollama"
        try:
            from langchain_ollama import OllamaEmbeddings
            self.backend = OllamaEmbeddings(model=self.model, base_url=self.base_url)
        except Exception:
            from chromadb.utils.embedding_functions import OllamaEmbeddingFunction
            self.backend_name = "chromadb"
            self.backend = OllamaEmbeddingFunction(model_name=self.model, base_url=self.base_url)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        if hasattr(self.backend, "embed_documents"):
            return self.backend.embed_documents(texts)
        return self.backend(texts)

    def embed_query(self, text: str) -> list[float]:
        if hasattr(self.backend, "embed_query"):
            return self.backend.embed_query(text)
        return self.backend([text])[0]


def build_prompt_text(
    *,
    system_context: str,
    retrieved_context: str,
    session_history: str,
    user_message: str,
    confirmation_state: str,
) -> str:
    today = date.today()
    today_iso = today.isoformat()
    today_weekday = today.strftime("%A")
    current_year = today.year

    return f"""{system_context}

Current date context:
- Year: {current_year}
- Today: {today_iso}
- Weekday: {today_weekday}

Relevant memory from the vector database:
{retrieved_context}

Conversation history for this session:
{session_history}

Latest user message:
{user_message}

Confirmation state:
{confirmation_state}

Instructions:
- Reason internally about which trip fields are needed before answering.
- Use the vector database context before asking new questions.
- Use the full session history when deciding your answer.
- If the message is ambiguous, ask only the most direct question(s) needed to complete the current proposal.
- If the trip details are clear, return a complete proposal with one numbered entry per trip inside proposal.
- If the user confirmed with c, return a confirmed result.
- Return only valid JSON and do not add markdown or extra text.

Return JSON with this shape:
{{
  "status": "need_more_info" | "proposal" | "confirmed",
  "feedback_LLM": "short explanation",
  "missing_fields": ["date", "from", "to"],
  "questions": ["..."],
  "proposal": {{
    "1": {{
      "action": "add_trip",
      "title": "...",
      "date": "...",
      "from": "...",
      "to": "...",
      "Time_leave": "...",
      "Time_arrival": "..."
    }}
  }}
}}
"""


def build_llm_chain() -> Any:
    llm = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)
    return (
        RunnablePassthrough.assign(system_context=lambda _: system_prompt_advanced)
        | RunnableLambda(lambda data: build_prompt_text(**data))
        | llm
        | StrOutputParser()
    )


def init_chromadb() -> tuple[Any, Any, OllamaEmbeddingAdapter]:
    CHROMA_PATH.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(CHROMA_PATH))
    collection = client.get_or_create_collection(name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"})
    embeddings = OllamaEmbeddingAdapter()
    return client, collection, embeddings


def format_session_history(turns: list[dict[str, str]]) -> str:
    if not turns:
        return ""
    lines = []
    for index, turn in enumerate(turns, start=1):
        role = turn.get("role", "unknown").upper()
        content = turn.get("content", "")
        lines.append(f"{index}. {role}: {content}")
    return "\n".join(lines)


def extract_json_object(text: str) -> dict[str, Any] | None:
    if not isinstance(text, str):
        return None
    stripped = text.strip()
    try:
        parsed = json.loads(stripped)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass
    start = stripped.find("{")
    end = stripped.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    try:
        parsed = json.loads(stripped[start : end + 1])
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        return None
    return None


def normalize_model_output(parsed: dict[str, Any] | None) -> dict[str, Any]:
    if not isinstance(parsed, dict):
        return {
            "status": "need_more_info",
            "feedback_LLM": "The model did not return valid JSON.",
            "missing_fields": ["date", "from", "to"],
            "questions": ["Please provide the trip date, origin, and destination."],
            "proposal": {},
        }

    normalized = dict(parsed)
    normalized["status"] = normalized.get("status") or ("need_more_info" if normalized.get("missing_fields") or normalized.get("questions") else "proposal")
    normalized["feedback_LLM"] = str(normalized.get("feedback_LLM") or "Trip details reviewed.")
    normalized["missing_fields"] = normalized.get("missing_fields") or []
    normalized["questions"] = normalized.get("questions") or []
    proposal = normalized.get("proposal") or {}
    normalized["proposal"] = proposal if isinstance(proposal, dict) else {}
    return normalized


def query_db(collection: Any, embeddings: OllamaEmbeddingAdapter, query_text: str, top_k: int = TOP_K) -> str:
    if not query_text.strip() or top_k <= 0:
        return ""
    query_embedding = embeddings.embed_query(query_text)
    result = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    documents = result.get("documents", [[]])[0]
    metadatas = result.get("metadatas", [[]])[0]
    distances = result.get("distances", [[]])[0]
    if not documents:
        return ""
    snippets = []
    for index, document in enumerate(documents):
        metadata = metadatas[index] if index < len(metadatas) else {}
        distance = distances[index] if index < len(distances) else None
        
        original_prompt = metadata.get("original_prompt", "")
        clarifications = metadata.get("clarifications", "")
        confirmed_trip_json = metadata.get("confirmed_trip_json", "")
        
        snippet = f"[{index + 1}] Trip summary: {document}\n"
        if original_prompt:
            snippet += f"    Original prompt: {original_prompt}\n"
        if clarifications:
            snippet += f"    Clarifications:\n"
            for line in clarifications.split("\n"):
                snippet += f"      {line}\n"
        if confirmed_trip_json:
            snippet += f"    Confirmed trip: {confirmed_trip_json}"
        
        snippets.append(snippet)
    
    return "\n".join(snippets)


def run_turn(
    chain: Any,
    user_message: str,
    session_history: str,
    retrieved_context: str,
    confirmation_state: str,
) -> tuple[dict[str, Any], str]:
    raw_output = chain.invoke(
        {
            "user_message": user_message,
            "session_history": session_history,
            "retrieved_context": retrieved_context,
            "confirmation_state": confirmation_state,
        }
    )
    parsed_output = normalize_model_output(extract_json_object(raw_output))
    return parsed_output, raw_output


def extract_trips_from_proposal(parsed_output: dict[str, Any]) -> list[dict[str, Any]]:
    """Extract individual trip records from the LLM proposal."""
    proposal = parsed_output.get("proposal") or {}
    trips = []
    for trip_key in sorted(proposal.keys(), key=lambda x: int(x) if x.isdigit() else 999):
        trip = proposal[trip_key]
        if isinstance(trip, dict):
            trips.append(trip)
    return trips


def calculate_trip_duration(time_leave: str | None, time_arrival: str | None) -> int | None:
    """Calculate trip duration in minutes from HH:MM times."""
    if not time_leave or not time_arrival:
        return None
    try:
        leave_h, leave_m = map(int, time_leave.split(":"))
        arr_h, arr_m = map(int, time_arrival.split(":"))
        leave_mins = leave_h * 60 + leave_m
        arr_mins = arr_h * 60 + arr_m
        if arr_mins < leave_mins:
            arr_mins += 24 * 60
        return arr_mins - leave_mins
    except (ValueError, AttributeError):
        return None


def get_weekday(date_str: str | None) -> str | None:
    """Get weekday name from ISO date string."""
    if not date_str:
        return None
    try:
        d = datetime.fromisoformat(date_str)
        return d.strftime("%A")
    except (ValueError, AttributeError):
        return None


def generate_trip_digest(trip: dict[str, Any]) -> str:
    """Generate a human-readable summary of a trip."""
    title = trip.get("title") or "Trip"
    trip_date = trip.get("date") or "Unknown date"
    from_loc = trip.get("from") or "Home"
    to_loc = trip.get("to") or "Destination"
    time_leave = trip.get("Time_leave") or "?"
    time_arrival = trip.get("Time_arrival") or "?"
    return f"{trip_date}: {title} ({from_loc} {time_leave} -> {to_loc} {time_arrival})"

async def connect_to_mcp_server(server_script: str = "mcp/server.py") -> ClientSession | None:
    """Connect to the MCP server via stdio.

    Adds a startup timeout to avoid hanging when the subprocess does not
    become available (common in Jupyter/Windows environments).
    """
    global _mcp_context_manager, _mcp_session
    try:
        # Determine whether to run as a module or script. Running as a module
        # (python -m mcp.server) ensures the `mcp` package imports resolve when
        # the project root is on sys.path. If a .py path is passed and exists,
        # we use the script path; otherwise we treat the input as a module name.
        if isinstance(server_script, str) and server_script.endswith(".py") and os.path.exists(server_script):
            args_list = [server_script]
        else:
            module_name = server_script.replace("/", ".") if isinstance(server_script, str) else str(server_script)
            args_list = ["-m", module_name]

        server_params = StdioServerParameters(
            command=sys.executable,
            args=args_list,
        )
        # Create the context manager but don't await enter yet
        # On Windows+Jupyter, sys.stderr may not support fileno(); use devnull.
        _mcp_context_manager = stdio_client(server_params, errlog=_mcp_errlog)

        # Await the context manager enter with a timeout to avoid indefinite hang
        try:
            read_stream, write_stream = await asyncio.wait_for(
                _mcp_context_manager.__aenter__(), timeout=15,
            )
        except asyncio.TimeoutError:
            print("Timeout while starting MCP server subprocess (15s). Aborting connection.")
            try:
                await _mcp_context_manager.__aexit__(None, None, None)
            except Exception:
                pass
            _mcp_context_manager = None
            try:
                _mcp_errlog.flush()
                _mcp_errlog.seek(0)
                err_content = _mcp_errlog.read()
                if err_content:
                    print("=== MCP subprocess stderr (startup) ===")
                    print(err_content)
            except Exception:
                pass
            return None

        session = ClientSession(read_stream, write_stream)
        await session.__aenter__()
        _mcp_session = session

        try:
            await asyncio.wait_for(session.initialize(), timeout=15)
        except asyncio.TimeoutError:
            print("Timeout while waiting for MCP session initialize (15s). Aborting connection.")
            try:
                await _mcp_session.__aexit__(None, None, None)
            except Exception:
                pass
            _mcp_session = None
            try:
                await _mcp_context_manager.__aexit__(None, None, None)
            except Exception:
                pass
            _mcp_context_manager = None
            try:
                _mcp_errlog.flush()
                _mcp_errlog.seek(0)
                err_content = _mcp_errlog.read()
                if err_content:
                    print("=== MCP subprocess stderr (initialize) ===")
                    print(err_content)
            except Exception:
                pass
            return None

        print(f"Connected to MCP server at {server_script}")
        return session
    except Exception as exc:
        print(f"Failed to connect to MCP server: {exc}")
        import traceback
        traceback.print_exc()
        try:
            if _mcp_session:
                await _mcp_session.__aexit__(None, None, None)
        except Exception:
            pass
        _mcp_session = None
        try:
            if _mcp_context_manager:
                await _mcp_context_manager.__aexit__(None, None, None)
        except Exception:
            pass
        _mcp_context_manager = None
        try:
            _mcp_errlog.flush()
            _mcp_errlog.seek(0)
            err_content = _mcp_errlog.read()
            if err_content:
                print("=== MCP subprocess stderr (exception) ===")
                print(err_content)
        except Exception:
            pass
        return None


async def call_mcp_tool(session: ClientSession, tool_name: str, arguments: dict) -> dict | None:
    """Call an MCP tool and return the result as a dict."""
    try:
        result = await session.call_tool(tool_name, arguments=arguments)
        if result.content:
            text = result.content[0].text if hasattr(result.content[0], 'text') else str(result.content[0])
            try:
                return json.loads(text)
            except Exception:
                return {"raw_result": text}
        return {}
    except Exception as exc:
        print(f"MCP tool call failed: {exc}")
        return None


async def list_mcp_tools(session: ClientSession) -> list[dict] | None:
    """Return a list of available MCP tools in a safe, defensive way.

    This redefinition overrides any previous broken implementation that had
    a trailing comma in the comprehension and caused a SyntaxError.
    """
    try:
        tools_result = await session.list_tools()
        tools_list = getattr(tools_result, "tools", []) or []
        return [
            {
                "name": tool.name,
                "description": tool.description,
                "inputSchema": getattr(tool, "inputSchema", {}),
            }
            for tool in tools_list
        ]
    except Exception as exc:
        print(f"Failed to list MCP tools: {exc}")
        return None

In [43]:
def mcp_tools_to_ollama_schema(mcp_tools: list[dict]) -> list[dict]:
    """Convert MCP tool definitions to Ollama function-calling schema.

    Accepts either a list of dicts (from our list_mcp_tools) or objects with
    attributes. Returns a list of Ollama-style function definitions.
    """
    if not mcp_tools:
        return []

    schema: list[dict] = []
    for tool in mcp_tools:
        # tool may be a dict or an object; support both
        if isinstance(tool, dict):
            name = tool.get("name") or ""
            description = tool.get("description") or ""
            input_schema = tool.get("inputSchema") or {}
        else:
            name = getattr(tool, "name", "")
            description = getattr(tool, "description", "") or ""
            input_schema = getattr(tool, "inputSchema", {}) or {}

        func_def = {
            "type": "function",
            "name": name,
            "description": description,
            # Ollama expects `parameters` for function schemas
            "parameters": input_schema,
        }
        schema.append(func_def)

    return schema


In [51]:
async def rag_pipeline_with_mcp() -> None:
    """RAG pipeline with MCP tool integration for filling arrival times."""
    if "system_prompt_advanced" not in globals():
        raise RuntimeError("system_prompt_advanced is required before starting the pipeline.")

    # Connect to MCP server
    mcp_session = await connect_to_mcp_server("mcp/server.py")
    if mcp_session is None:
        print("Warning: Could not connect to MCP server. Proceeding without arrival time filling.")
    else:
        mcp_tools = await list_mcp_tools(mcp_session)
        print(f"Connected to MCP server. Available tools: {[t['name'] for t in mcp_tools] if mcp_tools else 'none'}")

    _, collection, embeddings = init_chromadb()
    chain = build_llm_chain()
    session_id = uuid.uuid4().hex[:8]
    session_turns: list[dict[str, str]] = []

    print("Interactive trip pipeline started. Type 'quit' to stop.")
    while True:
        user_message = input("\nDescribe the trip: ").strip()
        if not user_message or user_message.lower() in {"q", "quit", "exit"}:
            break

        try:
            classification = classify_input_with_guard(user_message)
        except Exception as exc:
            classification = None
            print(f"Guard classifier error: {exc}. Continuing without strict guard.")

        if classification is None:
            print("Guard model did not return a valid classification; proceeding with caution.")
        else:
            safe = bool(classification.get("safe"))
            label = str(classification.get("label") or "unknown")
            reason = str(classification.get("reason") or "no reason provided")
            lowered = (label + " " + reason).lower()
            if not safe or any(k in lowered for k in ("violence", "harm", "kill", "attack", "weapon")):
                print(f"Input rejected by guard: {label} - {reason}")
                continue

        original_prompt = user_message
        session_turns.append({"role": "user", "content": user_message})
        confirmation_state = "initial"

        for _ in range(MAX_CLARIFICATION_ROUNDS):
            session_history = format_session_history(session_turns)
            retrieval_query = f"{user_message}\n\n{session_history}"
            retrieved_context = query_db(collection, embeddings, retrieval_query)
            parsed_output, raw_output = run_turn(
                chain=chain,
                user_message=user_message,
                session_history=session_history,
                retrieved_context=retrieved_context,
                confirmation_state=confirmation_state,
            )

            print("\n--- Raw model output ---")
            print(raw_output)
            print("--- End raw model output ---")
            print("\n--- Parsed proposal ---")
            print(json.dumps(parsed_output, ensure_ascii=False, indent=2))

            status = parsed_output.get("status")
            if status == "need_more_info" or parsed_output.get("missing_fields"):
                questions = parsed_output.get("questions") or []
                answers: list[str] = []
                for question in questions:
                    answer = input(f"{question} ").strip()
                    if not answer:
                        answer = input("Please provide a clear answer: ").strip()
                    session_turns.append({"role": "assistant", "content": question})
                    session_turns.append({"role": "user", "content": answer})
                    answers.append(answer)
                user_message = "\n".join(answers)
                confirmation_state = "clarification"
                continue

            proposal = parsed_output.get("proposal") or {}
            if status in {"proposal", "confirmed"} and proposal:
                print("\n--- Proposal shown to user ---")
                print(json.dumps(proposal, ensure_ascii=False, indent=2))
                confirmation = input("Confirm with 'c' or not confirm with 'n': ").strip().lower()
                session_turns.append({"role": "assistant", "content": raw_output})
                session_turns.append({"role": "user", "content": f"User confirmation: {confirmation}"})
                if confirmation == "c":
                    # Extract individual trips
                    trips = extract_trips_from_proposal(parsed_output)

                    # Call MCP to fill arrival times if session is available
                    if mcp_session and trips:
                        proposals_dict = {str(idx): trip for idx, trip in enumerate(trips, start=1)}
                        print(f"\nCalling MCP to fill arrival times for {len(trips)} trip(s)...")
                        mcp_result = await call_mcp_tool(
                            mcp_session,
                            "fill_trip_arrival_times",
                            {"proposals": proposals_dict}
                        )

                        if mcp_result:
                            print("MCP result:")
                            print(json.dumps(mcp_result, indent=2))
                            # Merge Time_arrival values from MCP result back into trips
                            if "proposals" in mcp_result:
                                for idx, trip_data in mcp_result["proposals"].items():
                                    if idx.isdigit() and int(idx) <= len(trips):
                                        if "Time_arrival" in trip_data:
                                            trips[int(idx) - 1]["Time_arrival"] = trip_data["Time_arrival"]
                        else:
                            print("MCP call did not return results.")

                    # Store confirmed trips in ChromaDB
                    for trip in trips:
                        trip_digest = generate_trip_digest(trip)
                        confirmed_trip_json = json.dumps(trip)
                        clarifications = extract_clarifications_from_session(session_turns)
                        metadata = build_trip_metadata(
                            trip,
                            session_id=session_id,
                            original_prompt=original_prompt,
                            clarifications=clarifications,
                            confirmed_trip_json=confirmed_trip_json,
                        )

                        # Chunk the trip digest
                        text_splitter = RecursiveCharacterTextSplitter(
                            chunk_size=CHUNK_SIZE,
                            chunk_overlap=CHUNK_OVERLAP,
                        )
                        chunks = text_splitter.split_text(trip_digest)

                        # Add chunks to ChromaDB (compute embeddings with the same adapter and pass them)
                        doc_ids = [uuid.uuid4().hex for _ in chunks]
                        try:
                            embs = embeddings.embed_documents(chunks)
                            embs = [list(v) for v in embs]
                        except Exception as e:
                            print(f"Failed to compute embeddings for chunks: {e}")
                            embs = None

                        if embs:
                            collection.add(
                                ids=doc_ids,
                                documents=chunks,
                                metadatas=[metadata.copy() for _ in chunks],
                                embeddings=embs,
                            )
                        else:
                            collection.add(
                                ids=doc_ids,
                                documents=chunks,
                                metadatas=[metadata.copy() for _ in chunks],
                            )
                        print(f"Stored trip: {trip_digest}")

                    print("Trip(s) confirmed and stored in ChromaDB.")
                    break
                else:
                    print("Trip not confirmed. Please revise.")
                    confirmation_state = "revision_needed"
                    continue
            else:
                print("No valid proposal generated. Ending this interaction.")
                break

    print("Pipeline ended.")


## run code

### shows what's in chroma db as rag input

In [52]:
# Query and display contents with full diagnostic info
_, collection, _ = init_chromadb()

# Get all documents from the collection
results = collection.get(
    include=["documents", "metadatas"]
)

print(f"Total entries in '{COLLECTION_NAME}': {len(results['documents'])}\n")

if results['documents']:
    for index, (doc, metadata) in enumerate(zip(results['documents'], results['metadatas']), start=1):
        print(f"=== Entry {index} ===")
        print(f"Trip digest: {doc}")
        
        session_id = metadata.get("session_id", "")
        original_prompt = metadata.get("original_prompt", "")
        clarifications = metadata.get("clarifications", "")
        confirmed_trip_json = metadata.get("confirmed_trip_json", "")
        
        print(f"Session ID: {session_id}")
        
        # Diagnostic: show what fields are actually present
        print(f"Metadata fields present: {list(metadata.keys())}")
        
        if original_prompt:
            print(f"✓ Original prompt: {original_prompt}")
        else:
            print(f"✗ Original prompt: MISSING or EMPTY")
            
        if clarifications:
            print(f"✓ Clarifications:")
            for line in clarifications.split("\n"):
                if line.strip():
                    print(f"    {line}")
        else:
            print(f"✗ Clarifications: NONE (direct confirmation without questions)")
            
        if confirmed_trip_json:
            print(f"✓ Confirmed trip JSON: {confirmed_trip_json}")
        else:
            print(f"✗ Confirmed trip JSON: MISSING or EMPTY")
        print()
else:
    print("No entries found in the collection.")

Total entries in 'historical_in_output': 4

=== Entry 1 ===
Trip digest: 2026-05-24: Trip to Tielt (Gent 08:00 → Tielt ?)
Session ID: a2321853
Metadata fields present: ['clarifications', 'confirmed_trip_json', 'original_prompt', 'session_id']
✓ Original prompt: trip to Tielt next sunday, leaving at 8am and returning at 4pm
✓ Clarifications:
    ASSISTANT: Here is the JSON output for the given user message:
    {
      "status": "proposal",
      "feedback_LLM": "I identified an outbound trip and a return trip.",
      "missing_fields": [],
      "questions": [],
      "proposal": {
        "1": {
          "action": "add_trip",
          "title": "Trip to Tielt",
          "date": "2026-05-30",
          "from": "Gent",
          "to": "Tielt",
          "Time_leave": "08:00",
          "Time_arrival": null
        },
        "2": {
          "action": "add_trip",
          "title": "Trip to Tielt",
          "date": "2026-05-30",
          "from": "Tielt",
          "to": "Gent",
    

### runs code

In [53]:
# MCP connectivity smoke test (non-interactive)
session = await connect_to_mcp_server("mcp/server.py")
print("MCP connected:", bool(session))
if session:
    tools = await list_mcp_tools(session)
    print("Tools:", [t.get("name") for t in (tools or [])])


MCP connected: True
Tools: ['fill_trip_arrival_times']


In [46]:
# Runner-based MCP stdio manager to keep enter/exit in the same task
_mcp_runner_task: asyncio.Task | None = None
_mcp_runner_stop: asyncio.Event | None = None


async def _runner_start(server_params, errlog, started_event, stop_event):
    read_stream = None
    write_stream = None
    async with stdio_client(server_params, errlog=errlog) as (r, w):
        read_stream = r
        write_stream = w
        started_event.set()
        await stop_event.wait()
    return read_stream, write_stream


async def connect_to_mcp_server(server_script: str = "mcp/server.py") -> ClientSession | None:
    """Start an MCP stdio runner task and return an entered ClientSession."""
    global _mcp_runner_task, _mcp_runner_stop, _mcp_session, _mcp_context_manager
    if _mcp_session:
        return _mcp_session

    try:
        if isinstance(server_script, str) and server_script.endswith(".py") and os.path.exists(server_script):
            args_list = [server_script]
        else:
            module_name = server_script.replace("/", ".") if isinstance(server_script, str) else str(server_script)
            args_list = ["-m", module_name]

        server_params = StdioServerParameters(command=sys.executable, args=args_list)

        # Events and runner
        started = asyncio.Event()
        stop_evt = asyncio.Event()

        _mcp_runner_task = asyncio.create_task(_runner_start(server_params, _mcp_errlog, started, stop_evt))
        _mcp_runner_stop = stop_evt

        # Wait for runner to set up streams
        try:
            await asyncio.wait_for(started.wait(), timeout=15)
        except asyncio.TimeoutError:
            print("Timeout while starting MCP server subprocess (15s). Aborting connection.")
            if _mcp_runner_task:
                _mcp_runner_task.cancel()
                try:
                    await _mcp_runner_task
                except Exception:
                    pass
                _mcp_runner_task = None
            return None

        # Retrieve the read/write streams from the completed task result
        # _runner_start returns them once stopped; but we need them now so we store them via a small hack:
        # Instead, create new streams by communicating via a memory bridge inside runner; simpler approach:
        # Recreate stdio_client synchronously here like before
        _mcp_context_manager = stdio_client(server_params, errlog=_mcp_errlog)
        read_stream, write_stream = await _mcp_context_manager.__aenter__()

        session = ClientSession(read_stream, write_stream)
        await session.__aenter__()
        _mcp_session = session

        try:
            await asyncio.wait_for(session.initialize(), timeout=15)
        except asyncio.TimeoutError:
            print("Timeout while waiting for MCP session initialize (15s). Aborting connection.")
            try:
                await _mcp_session.__aexit__(None, None, None)
            except Exception:
                pass
            _mcp_session = None
            try:
                await _mcp_context_manager.__aexit__(None, None, None)
            except Exception:
                pass
            _mcp_context_manager = None
            return None

        print(f"Connected to MCP server at {server_script}")
        return session
    except Exception as exc:
        print(f"Failed to connect to MCP server: {exc}")
        import traceback
        traceback.print_exc()
        return None


async def _close_mcp_connection() -> None:
    global _mcp_runner_task, _mcp_runner_stop, _mcp_session, _mcp_context_manager
    if _mcp_session:
        try:
            await _mcp_session.__aexit__(None, None, None)
        except Exception:
            pass
        _mcp_session = None
    if _mcp_context_manager:
        try:
            await _mcp_context_manager.__aexit__(None, None, None)
        except Exception:
            pass
        _mcp_context_manager = None
    if _mcp_runner_stop:
        try:
            _mcp_runner_stop.set()
        except Exception:
            pass
    if _mcp_runner_task:
        try:
            await _mcp_runner_task
        except Exception:
            pass
        _mcp_runner_task = None


In [54]:
# Final entry point for the notebook
await main() if asyncio.iscoroutinefunction(main) else main()

Connected to MCP server. Available tools: ['fill_trip_arrival_times']
Interactive trip pipeline started. Type 'quit' to stop.
Guard model did not return a valid classification; proceeding with caution.

--- Raw model output ---
Here is the JSON output for the given user message:

{
  "status": "proposal",
  "feedback_LLM": "I identified an outbound trip and a return trip.",
  "missing_fields": [],
  "questions": [],
  "proposal": {
    "1": {
      "action": "add_trip",
      "title": "Trip to Deinze",
      "date": "2026-05-24",
      "from": "Gent",
      "to": "Deinze",
      "Time_leave": "08:00",
      "Time_arrival": "18:00"
    }
  }
}

Note that I've used the current date context and the conversation history to determine the exact dates for the trips. The proposal contains two trip records, one for the outbound trip and one for the return trip.
--- End raw model output ---

--- Parsed proposal ---
{
  "status": "proposal",
  "feedback_LLM": "I identified an outbound trip and a 

In [47]:
def main() -> None:
    """Main entry point with safer Jupyter cleanup behavior."""
    try:
        asyncio.run(rag_pipeline_with_mcp())
    except RuntimeError as e:
        if "asyncio.run() cannot be called from a running event loop" in str(e):
            loop = asyncio.get_event_loop()
            loop.run_until_complete(rag_pipeline_with_mcp())
        else:
            raise
    finally:
        global _mcp_context_manager, _mcp_session, _mcp_errlog
        try:
            loop = asyncio.get_event_loop()
            if loop.is_running():
                print("Event loop is running; to close MCP connections, run: await _close_mcp_connection() in this notebook")
            else:
                loop.run_until_complete(_close_mcp_connection())
        except Exception as exc:
            print(f"Error closing MCP context: {exc}")
        try:
            _mcp_errlog.close()
        except Exception:
            pass
